# Séance 7 — Introduction à PySpark

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 7.1 à 7.4

## Ce que vous saurez faire à la fin

- démarrer une session Spark locale et charger des données avec un schéma explicite ;
- traduire en PySpark les opérations Pandas que vous maîtrisez déjà ;
- lire un plan d'exécution et comprendre pourquoi rien ne se passe avant l'action ;
- mesurer le point de bascule entre Pandas et Spark, et le justifier.

## Préparation obligatoire avant la séance

```bash
pip install pyspark
python src/generate_data.py --big 5000000   # environ 400 Mo, comptez 3 à 5 minutes
```

Spark exige un environnement Java (JDK 8, 11 ou 17). Vérifiez avec `java -version`.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import csv
import os
import time

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

---
## Atelier 7.1 — Session Spark et schéma explicite (40 min)

### Partie guidée

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (DoubleType, IntegerType, LongType, StringType,
                               StructField, StructType, TimestampType)

spark = (SparkSession.builder
         .appName('PMD-M1')
         .master('local[*]')                            # tous les cœurs de la machine
         .config('spark.driver.memory', '4g')
         .config('spark.sql.shuffle.partitions', '16')  # défaut 200, trop pour du local
         # Conversion invalide -> null au lieu d'une exception (équivalent errors='coerce').
         # Indispensable sous Spark 4, où le mode ANSI est activé par défaut.
         .config('spark.sql.ansi.enabled', 'false')
         # Une date qui ne respecte pas le format -> null, sans SparkUpgradeException
         .config('spark.sql.legacy.timeParserPolicy', 'CORRECTED')
         .getOrCreate())

spark.sparkContext.setLogLevel('ERROR')
print('Spark', spark.version)
print('Cœurs disponibles :', spark.sparkContext.defaultParallelism)

**`spark.sql.shuffle.partitions` à 200 par défaut** est un réglage pensé pour un cluster.
En local il crée 200 tâches minuscules dont l'orchestration coûte plus cher que le calcul.
C'est la première cause de la réputation de lenteur de Spark sur petits volumes.

In [ ]:
import time

events_path = str(RAW_DIR / 'evenements_volumineux.csv')

# Méthode 1 : inférence automatique du schéma. Simple, mais Spark lit le fichier
# une première fois uniquement pour deviner les types.
start = time.perf_counter()
events_inferred = spark.read.csv(events_path, header=True, inferSchema=True)
print(f'Inférence : {time.perf_counter() - start:.1f} s')
events_inferred.printSchema()

In [ ]:
# Méthode 2 : schéma explicite. Une seule lecture, et aucune surprise de typage.
EVENTS_SCHEMA = StructType([
    StructField('id_evenement', LongType(), True),
    StructField('horodatage', TimestampType(), True),
    StructField('id_utilisateur', IntegerType(), True),
    StructField('categorie', StringType(), True),
    StructField('ville', StringType(), True),
    StructField('montant', DoubleType(), True),
    StructField('duree_s', IntegerType(), True),
])

start = time.perf_counter()
events = spark.read.csv(events_path, header=True, schema=EVENTS_SCHEMA)
print(f'Schéma explicite : {time.perf_counter() - start:.3f} s')

L'écart est spectaculaire, et il a une explication simple : avec un schéma explicite,
**rien n'a encore été lu**. Spark s'est contenté d'enregistrer une intention.

In [ ]:
# La preuve : count() est une ACTION, elle déclenche le calcul réel
start = time.perf_counter()
n_rows = events.count()
print(f'{n_rows} lignes comptées en {time.perf_counter() - start:.1f} s')

### Partie autonome

In [ ]:
# Q1. Charger le fichier ventes_brutes.csv dans Spark avec un schéma explicite.
#     Attention : prix_unitaire contient du texte pour une partie des lignes.
#     Quel type déclarez-vous, et pourquoi ?
#
# Réponse : prix_unitaire est déclaré en StringType.
# Déclarer DoubleType ne ferait PAS échouer la lecture : en mode PERMISSIVE (défaut),
# Spark remplace silencieusement par null toute valeur non convertible ("12,50 €", "N/A"...).
# On perdrait des prix pourtant récupérables, sans aucun message. Lu comme texte, le prix
# est converti dans une étape explicite du nettoyage (Q2), où les échecs sont comptés.
# Même logique pour tout le fichier brut : c'est la couche « bronze », on la lit telle
# quelle en texte et on type à l'étape suivante.

sales_raw_path = str(RAW_DIR / 'ventes_brutes.csv')

with open(sales_raw_path, encoding='utf-8-sig') as file:
    raw_columns = next(csv.reader(file))
print('Colonnes du fichier :', raw_columns)

SALES_RAW_SCHEMA = StructType([StructField(column, StringType(), True)
                               for column in raw_columns])
sales_raw = spark.read.csv(sales_raw_path, header=True, schema=SALES_RAW_SCHEMA)
sales_raw.printSchema()

# Preuve : ce que l'on perdrait en déclarant prix_unitaire en DoubleType
schema_double = StructType([
    StructField(column, DoubleType() if column == 'prix_unitaire' else StringType(), True)
    for column in raw_columns
])
as_double = spark.read.csv(sales_raw_path, header=True, schema=schema_double)
n_null_text = sales_raw.filter(F.col('prix_unitaire').isNull()).count()
n_null_double = as_double.filter(F.col('prix_unitaire').isNull()).count()
print(f'prix_unitaire vides -> lu en texte : {n_null_text} | lu en double : {n_null_double}')
print(f'=> {n_null_double - n_null_text} prix perdus silencieusement avec DoubleType')

---
## Atelier 7.2 — Rejouer le nettoyage en PySpark (55 min)

### Partie guidée : le dictionnaire de traduction

| Pandas | PySpark |
|---|---|
| `df[df.a > 5]` | `df.filter(F.col('a') > 5)` |
| `df['b'] = df['a'] * 2` | `df.withColumn('b', F.col('a') * 2)` |
| `df[['a','b']]` | `df.select('a', 'b')` |
| `df.rename(columns={'a':'c'})` | `df.withColumnRenamed('a', 'c')` |
| `df.groupby('a')['b'].sum()` | `df.groupBy('a').agg(F.sum('b'))` |
| `df.drop_duplicates()` | `df.dropDuplicates()` |
| `df.dropna(subset=['a'])` | `df.na.drop(subset=['a'])` |
| `df.fillna({'a': 0})` | `df.na.fill({'a': 0})` |
| `df.sort_values('a')` | `df.orderBy('a')` |
| `df.head(5)` | `df.show(5)` ou `df.limit(5).toPandas()` |

**La différence de fond n'est pas syntaxique.** En Pandas, chaque ligne s'exécute
immédiatement. En Spark, elle s'ajoute à un plan qui ne sera évalué qu'à la première action.

In [ ]:
clean_events = (events
                .filter(F.col('montant') > 0)
                .filter(F.col('duree_s').between(1, 7200))
                .withColumn('ville', F.lower(F.trim(F.col('ville'))))
                .withColumn('jour', F.to_date('horodatage'))
                .withColumn('heure', F.hour('horodatage'))
                .withColumn('est_weekend', F.dayofweek('horodatage').isin([1, 7]))
                .withColumn('tranche_montant',
                            F.when(F.col('montant') < 30, 'faible')
                             .when(F.col('montant') < 100, 'moyen')
                             .otherwise('eleve'))
                .dropDuplicates(['id_evenement']))

print('Aucun calcul lancé à ce stade.')
clean_events.printSchema()

In [ ]:
# Le plan d'exécution : ce que Spark VA faire, et comment Catalyst l'a réorganisé
clean_events.explain(mode='formatted')

Repérez dans le plan la remontée des filtres au plus près de la lecture du fichier
(*predicate pushdown*) et la sélection des seules colonnes utiles (*column pruning*).
Vous n'avez pas demandé ces optimisations : Catalyst les a déduites de votre plan.
C'est la contrepartie de l'évaluation paresseuse.

### Partie autonome

In [ ]:
# Q2. Écrire l'équivalent PySpark du nettoyage de la séance 3 sur les ventes :
#     conversion du prix textuel, normalisation des colonnes texte, suppression des
#     doublons, parsing de la date, calcul du montant.
#     Indices : F.regexp_replace, F.to_timestamp, F.coalesce.

def clean_text(column, lowercase=True):
    """trim (+ minuscules), chaîne vide -> null."""
    value = F.trim(F.col(column))
    if lowercase:
        value = F.lower(value)
    return F.when(value == '', None).otherwise(value)


def parse_number(column):
    """'12,50 €', ' 12.5 EUR ', '1 299,00', '10 %' -> double ; texte non numérique -> null."""
    value = F.regexp_replace(F.col(column), r'[^0-9,.\-]', '')   # retire €, EUR, %, espaces
    value = F.regexp_replace(value, ',', '.')                     # virgule décimale -> point
    return F.when(value == '', None).otherwise(value).cast(DoubleType())


DATE_FORMATS = ['yyyy-MM-dd HH:mm:ss', 'yyyy-MM-dd HH:mm', 'yyyy-MM-dd',
                'dd/MM/yyyy HH:mm:ss', 'dd/MM/yyyy HH:mm', 'dd/MM/yyyy',
                'yyyy/MM/dd', 'dd-MM-yyyy']


def parse_date(column):
    """Essaie chaque format et garde le premier qui réussit (les autres renvoient null)."""
    value = F.trim(F.col(column))
    return F.coalesce(*[F.to_timestamp(value, fmt) for fmt in DATE_FORMATS])


ID_COLUMNS = [c for c in ['id_commande', 'id_client', 'id_produit', 'id_magasin'] if c in raw_columns]
TEXT_COLUMNS = [c for c in ['categorie', 'canal', 'statut', 'ville'] if c in raw_columns]

# 1. doublons stricts (ligne identique)
sales_spark = sales_raw.dropDuplicates()

# 2. normalisation texte : identifiants nettoyés sans changer la casse, libellés en minuscules
for column in ID_COLUMNS:
    sales_spark = sales_spark.withColumn(column, clean_text(column, lowercase=False))
for column in TEXT_COLUMNS:
    sales_spark = sales_spark.withColumn(column, clean_text(column))

# 3. conversions de types
sales_spark = (sales_spark
               .withColumn('prix_unitaire', parse_number('prix_unitaire'))
               .withColumn('quantite', parse_number('quantite').cast(IntegerType()))
               .withColumn('remise_pct', F.coalesce(parse_number('remise_pct'), F.lit(0.0)))
               .withColumn('date_commande', parse_date('date_commande')))

# 4. doublons sur la clé métier. Attention : dropDuplicates(['id_commande']) fusionnerait
#    TOUTES les lignes sans identifiant en une seule ; on ne dédoublonne donc que les clés présentes.
with_id = sales_spark.filter(F.col('id_commande').isNotNull()).dropDuplicates(['id_commande'])
without_id = sales_spark.filter(F.col('id_commande').isNull())
sales_spark = with_id.unionByName(without_id)

# 5. montants (remise exprimée en % (10) ou en fraction (0.10))
discount_rate = (F.when(F.col('remise_pct') > 1, F.col('remise_pct') / 100)
                  .otherwise(F.col('remise_pct')))
sales_spark = (sales_spark
               .withColumn('montant', F.round(F.col('quantite') * F.col('prix_unitaire'), 2))
               .withColumn('montant_net', F.round(F.col('montant') * (1 - discount_rate), 2))
               .cache())

# Bilan du nettoyage (première action : c'est ici que tout le plan s'exécute)
print('Lignes brutes   :', sales_raw.count())
print('Lignes nettoyées:', sales_spark.count())
(sales_spark
 .select(*[F.sum(F.col(c).isNull().cast('int')).alias(f'{c}_null')
           for c in ['prix_unitaire', 'quantite', 'date_commande', 'montant']])
 .show(truncate=False))
sales_spark.show(5, truncate=False)

**Q3.** Dans votre version PySpark, quelle opération du nettoyage Pandas avez-vous eu le
plus de mal à traduire ? Pourquoi ?

*Votre réponse :* le **parsing de la date**. En Pandas, `pd.to_datetime(..., errors='coerce')`
devine seul le format, ligne par ligne, et transforme les échecs en `NaT`. Spark n'a pas
d'équivalent : `F.to_timestamp` attend **un** format. Il faut donc lister explicitement les
formats rencontrés dans le fichier et les essayer l'un après l'autre avec `F.coalesce`.
Il faut aussi régler la session (`spark.sql.ansi.enabled=false`,
`timeParserPolicy=CORRECTED`) pour qu'une date invalide donne `null` au lieu de faire
échouer tout le job. C'est plus verbeux, mais plus sûr : aucun format n'est deviné à tort
(par exemple `03/04/2024` lu comme 4 mars au lieu de 3 avril).

Deuxième difficulté : la **déduplication sur la clé**. `dropDuplicates(['id_commande'])`
regroupe aussi toutes les lignes dont la clé est nulle, ce que `drop_duplicates` en Pandas
fait également mais qu'on remarque moins ; il a fallu séparer les lignes sans identifiant.

---
## Atelier 7.3 — Agrégations, jointures et partitions (45 min)

In [ ]:
summary = (clean_events
           .groupBy('ville', 'categorie')
           .agg(F.count('*').alias('nb_evenements'),
                F.round(F.sum('montant'), 2).alias('ca_total'),
                F.round(F.avg('montant'), 2).alias('montant_moyen'),
                F.expr('percentile_approx(montant, 0.5)').alias('montant_median'),
                F.countDistinct('id_utilisateur').alias('utilisateurs_uniques'))
           .orderBy(F.desc('ca_total')))

summary.show(10, truncate=False)

`percentile_approx` calcule une médiane **approchée**. Une médiane exacte imposerait de
trier l'intégralité des données sur l'ensemble des machines, ce qui coûte infiniment plus
cher. Sur des volumes de cet ordre, l'approximation est indiscernable du résultat exact.

In [ ]:
# Écriture partitionnée : chaque valeur de partition devient un sous-dossier
output_path = str(PROCESSED_DIR / 'evenements_spark')

(clean_events
 .select('id_evenement', 'jour', 'ville', 'categorie', 'montant', 'duree_s')
 .write.mode('overwrite')
 .partitionBy('categorie')
 .parquet(output_path))

print('Écrit dans', output_path)
for folder in sorted(os.listdir(output_path))[:6]:
    print(' ', folder)

Le partitionnement sur disque permet à une lecture filtrée sur `categorie` de ne toucher
que le sous-dossier concerné. Choisissez la clé de partitionnement en fonction des filtres
les plus fréquents, et évitez les clés de forte cardinalité : mille partitions de dix lignes
sont pires que pas de partitionnement du tout.

In [ ]:
# Q4. Reproduire en PySpark l'assemblage de la séance 6 : joindre les ventes
#     aux produits et aux magasins, puis produire le chiffre d'affaires par région
#     et par catégorie. Comparez le résultat à celui obtenu en Pandas.

products_sdf = spark.read.csv(str(RAW_DIR / 'produits.csv'), header=True, inferSchema=True)
stores_sdf = spark.read.csv(str(RAW_DIR / 'magasins.csv'), header=True, inferSchema=True)

REGION_COL = next((c for c in stores_sdf.columns if 'region' in c.lower()), None)
assert REGION_COL, f'colonne région introuvable dans magasins.csv : {stores_sdf.columns}'


def prepare_dimension(dimension, key, fact):
    """Clé convertie en texte nettoyé (même type que dans les faits), unicité garantie.
    Les colonnes déjà présentes dans les faits sont retirées : Spark n'ajoute pas de
    suffixe comme pandas, deux colonnes homonymes rendraient la suite ambiguë."""
    keep = [c for c in dimension.columns if c == key or c not in fact.columns]
    return (dimension.select(*keep)
            .withColumn(key, F.trim(F.col(key).cast(StringType())))
            .dropDuplicates([key]))          # équivalent de validate='many_to_one'


dim_products_sdf = prepare_dimension(products_sdf, 'id_produit', sales_spark)
dim_stores_sdf = prepare_dimension(stores_sdf, 'id_magasin', sales_spark)

n_before = sales_spark.count()
sales_with_dims = (sales_spark
                   .join(dim_products_sdf, on='id_produit', how='left')
                   .join(dim_stores_sdf, on='id_magasin', how='left'))
n_after = sales_with_dims.count()
print(f'Lignes : {n_before} -> {n_after}')
assert n_before == n_after, 'explosion de lignes à la jointure'

ca_region_category_spark = (sales_with_dims
    .withColumn(REGION_COL, F.coalesce(F.col(REGION_COL).cast(StringType()), F.lit('sans magasin')))
    .withColumn('categorie', F.coalesce(F.col('categorie'), F.lit('inconnue')))
    .groupBy(REGION_COL, 'categorie')
    .agg(F.round(F.sum('montant'), 2).alias('ca_spark'))
    .orderBy(REGION_COL, F.desc('ca_spark')))
ca_region_category_spark.show(20, truncate=False)

# --- Même calcul en Pandas, sur la sortie de la séance 6 --------------------------
sales_pd = load_dataset('ventes_transformees')
stores_pd = pd.read_csv(RAW_DIR / 'magasins.csv').drop_duplicates('id_magasin')
sales_pd = sales_pd.drop(columns=[REGION_COL], errors='ignore')

ca_region_category_pandas = (sales_pd
    .merge(stores_pd[['id_magasin', REGION_COL]], on='id_magasin', how='left',
           validate='many_to_one')
    .assign(**{REGION_COL: lambda d: d[REGION_COL].fillna('sans magasin').astype(str),
               'categorie': lambda d: d['categorie'].astype('string').str.strip()
                                                    .str.lower().fillna('inconnue')})
    .groupby([REGION_COL, 'categorie'], as_index=False)['montant'].sum()
    .rename(columns={'montant': 'ca_pandas'}))

# --- Comparaison ------------------------------------------------------------------
comparison_ca = ca_region_category_spark.toPandas().merge(
    ca_region_category_pandas, on=[REGION_COL, 'categorie'], how='outer')
comparison_ca['ecart'] = (comparison_ca['ca_spark'] - comparison_ca['ca_pandas']).round(2)

print(f"CA total Spark  : {comparison_ca['ca_spark'].sum():,.2f}")
print(f"CA total Pandas : {comparison_ca['ca_pandas'].sum():,.2f}")
print('Couples présents d\'un seul côté :',
      comparison_ca[['ca_spark', 'ca_pandas']].isna().any(axis=1).sum())
# Un écart signale une différence de NETTOYAGE (formats de date ou de prix traités
# différemment, doublons...), pas de jointure : les deux pipelines doivent appliquer
# exactement les mêmes règles pour donner le même résultat.
comparison_ca.sort_values('ecart', key=lambda s: s.abs(), ascending=False).head(10)

In [ ]:
# Q5. Exécuter .explain() sur votre jointure. Repérez le type de jointure choisi
#     par Spark (BroadcastHashJoin ou SortMergeJoin) et expliquez ce choix
#     au regard de la taille des tables.

print('Seuil de broadcast :', spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))
sales_with_dims.explain(mode='formatted')

# Contre-expérience : broadcast désactivé -> Spark est contraint au SortMergeJoin
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
print('\n--- Broadcast désactivé ---')
sales_spark.join(dim_products_sdf, on='id_produit', how='left').explain()
spark.conf.unset('spark.sql.autoBroadcastJoinThreshold')

# Réponse :
# Spark choisit un BroadcastHashJoin pour les deux jointures. produits.csv et magasins.csv
# font quelques Ko, bien en dessous du seuil spark.sql.autoBroadcastJoinThreshold (10 Mo par
# défaut). Spark copie donc la petite table entière sur chaque exécuteur, y construit une
# table de hachage, et chaque partition des ventes se joint localement : la grosse table
# n'est jamais redistribuée (pas de shuffle).
# Le SortMergeJoin n'est retenu que lorsque les deux côtés sont trop gros pour être diffusés :
# il faut alors redistribuer ET trier les deux tables sur la clé (Exchange + Sort dans le
# plan), ce qui est bien plus coûteux. La contre-expérience ci-dessus le montre.
# NB : avec l'exécution adaptative (AQE, « AdaptiveSparkPlan isFinalPlan=false »), le plan
# affiché est provisoire ; Spark peut encore convertir un SortMergeJoin en broadcast à
# l'exécution, une fois les vraies tailles connues.

---
## Atelier 7.4 — Où se situe le point de bascule (40 min)

C'est l'atelier dont vous devrez rendre compte en soutenance. La question *pourquoi avoir
utilisé Spark ici ?* appelle une réponse chiffrée, pas une intuition.

In [ ]:
def benchmark_pandas(path, n_rows):
    start = time.perf_counter()
    df = pd.read_csv(path, nrows=n_rows)
    result = (df[df['montant'] > 0]
              .groupby(['ville', 'categorie'])
              .agg(nb=('id_evenement', 'count'), ca=('montant', 'sum')))
    duration = time.perf_counter() - start
    memory_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    del df
    return duration, memory_mb, len(result)


def benchmark_spark(path, n_rows):
    start = time.perf_counter()
    df = spark.read.csv(path, header=True, schema=EVENTS_SCHEMA).limit(n_rows)
    result = (df.filter(F.col('montant') > 0)
              .groupBy('ville', 'categorie')
              .agg(F.count('*').alias('nb'), F.sum('montant').alias('ca')))
    n_groups = result.count()
    return time.perf_counter() - start, None, n_groups

In [ ]:
SAMPLE_SIZES = [50_000, 250_000, 1_000_000, 3_000_000]
measurements = []

for n_rows in SAMPLE_SIZES:
    pandas_duration, memory_mb, _ = benchmark_pandas(events_path, n_rows)
    spark_duration, _, _ = benchmark_spark(events_path, n_rows)
    measurements.append({
        'lignes': n_rows,
        'pandas_s': round(pandas_duration, 2),
        'spark_s': round(spark_duration, 2),
        'memoire_pandas_Mo': round(memory_mb, 1),
        'rapport': round(pandas_duration / spark_duration, 2),
    })
    print(measurements[-1])

comparison = pd.DataFrame(measurements)
comparison

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(comparison['lignes'], comparison['pandas_s'], marker='o', label='pandas')
ax.plot(comparison['lignes'], comparison['spark_s'], marker='s', label='pyspark')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('nombre de lignes')
ax.set_ylabel('durée (s)')
ax.set_title('Point de bascule Pandas / PySpark')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Interprétation attendue

Sur une machine unique, **Pandas gagne presque toujours en durée brute**. Spark paie un coût
fixe de démarrage et de coordination que le parallélisme local ne compense pas.

L'intérêt de Spark n'est donc pas la vitesse à cette échelle, mais :

- la capacité à traiter un volume qui ne tient pas en mémoire ;
- la possibilité de passer du local au cluster sans réécrire le code ;
- la lecture sélective de données partitionnées, sans charger le reste.

**Répondre « j'ai utilisé Spark parce que c'est plus rapide » en soutenance est une réponse
fausse et sera sanctionnée.** La bonne réponse porte sur le volume, la mémoire disponible
et la trajectoire du projet.

In [ ]:
# Q6. Compléter le tableau : à partir de quel volume Pandas dépasse-t-il la mémoire
#     disponible sur VOTRE machine ? Estimez-le à partir de la colonne memoire_pandas_Mo
#     et de la RAM dont vous disposez. Écrivez la valeur et le raisonnement.

try:
    import psutil
    total_ram_mb = psutil.virtual_memory().total / 1024 ** 2
    available_ram_mb = psutil.virtual_memory().available / 1024 ** 2
except ImportError:
    try:  # Linux / macOS
        total_ram_mb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024 ** 2
    except (AttributeError, ValueError):  # Windows sans psutil : renseigner à la main
        total_ram_mb = 8 * 1024
    available_ram_mb = total_ram_mb * 0.6

# La mémoire Pandas croît linéairement avec le nombre de lignes : on estime le coût par ligne
mb_per_row, intercept = np.polyfit(comparison['lignes'], comparison['memoire_pandas_Mo'], 1)

# Un DataFrame ne doit pas occuper plus d'environ 1/3 de la RAM disponible : filtres,
# groupby et merge créent des copies intermédiaires (2 à 3 fois la taille de départ).
SAFETY_FACTOR = 3
max_rows_load = (available_ram_mb - intercept) / mb_per_row
max_rows_safe = max_rows_load / SAFETY_FACTOR

print(f'RAM totale        : {total_ram_mb / 1024:.1f} Go')
print(f'RAM disponible    : {available_ram_mb / 1024:.1f} Go')
print(f'Coût Pandas       : {mb_per_row * 1e6:.0f} Mo par million de lignes')
print(f'Limite de chargement brut        : ~{max_rows_load / 1e6:.0f} millions de lignes')
print(f'Limite de travail confortable    : ~{max_rows_safe / 1e6:.0f} millions de lignes')

# Raisonnement à recopier en soutenance (avec VOS chiffres) :
# - la mémoire mesurée est proportionnelle au nombre de lignes : X Mo par million ;
# - avec Y Go disponibles, le chargement seul sature vers N millions de lignes ;
# - en pratique, les traitements dupliquent les données : au-delà de ~N/3 millions de
#   lignes, Pandas swappe ou plante (MemoryError). C'est à partir de ce volume que Spark
#   devient nécessaire : il traite les données par partitions sans tout charger en mémoire.

In [ ]:
spark.stop()
print('Session fermée.')

---
## Livrable de la séance

- un script `src/spark_pipeline.py` autonome, exécutable par `spark-submit`, produisant
  un Parquet partitionné ;
- le tableau comparatif chiffré Pandas / PySpark et le graphique associé ;
- un paragraphe écrit justifiant, pour votre projet, le choix de traiter telle étape en
  Spark plutôt qu'en Pandas.